# Vachan V2 spike — a Hinglish *tone dial* with control vectors

**What this proves:** you can shift a model's tone (formal English ↔ casual Hinglish) by *adding a direction to its hidden states at generation time* — **no retraining, no fine-tuning**. This is the V2 path beyond prompt-steering.

**How it works (plain English):** a control vector is a single direction in the model's "thought space". We find it by showing the model the *same* situation described two ways — once as a casual Hinglish friend, once as a formal corporate email — and subtracting the two internal states. That difference *is* the tone axis. At generation we add `coeff × vector`: `+` pushes Hinglish, `-` pushes formal English, `0` is the untouched model.

**n8n analogy:** the model is a pipeline; the control vector is a slider node we splice into the middle layers that nudges every token toward one tone.

> Runtime: ~5–10 min on a Kaggle **T4** (set Accelerator → GPU T4 first). Llama-3.1-8B in 4-bit ≈ 6 GB, fits one T4.

## 0. Setup (read me)

1. **Kaggle**: top-right **⋮ → Accelerator → GPU T4 x2** (or just T4). Also **Internet: On** (Settings).
2. **Model**: we use a **non-gated** Llama-3.1-8B mirror, so you need **no HuggingFace token**. (If you'd rather use the official `meta-llama/Llama-3.1-8B-Instruct`, accept its license on HF, make a read token, add it in Kaggle **Add-ons → Secrets** as `HF_TOKEN`, and flip `MODEL` below.)
3. Run cells top to bottom (**Run All**).

In [ ]:
# repeng = the control-vector library (extract + inject). The rest are standard.
!pip install -q repeng transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from repeng import ControlVector, ControlModel, DatasetEntry

# Non-gated mirror — no HF login needed. Same weights as Meta's official release.
MODEL = "NousResearch/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token_id = tokenizer.eos_token_id

# 4-bit so 8B fits a single 16 GB T4 (≈6 GB of weights).
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")

# Wrap a band of middle/late layers — these are where 'style' lives. ControlModel
# lets us inject a vector into exactly these layers at generation time.
model = ControlModel(model, list(range(-5, -18, -1)))
print("loaded:", MODEL)

## 1. Define the tone axis with contrastive pairs

We hand `repeng` many `(positive, negative)` pairs that are identical *except* for the tone instruction. It reads the model's hidden state for each and learns the single direction that separates them. The short generic suffixes just give it many token positions to read from — more positions → a cleaner vector.

In [ ]:
# The two ends of the dial:
POS = "You are texting a close Indian friend on WhatsApp: warm, casual, code-mix Hindi and English (Hinglish), short and natural."
NEG = "You are writing a formal corporate email: polished, professional, pure English, complete sentences."

# Generic continuations — the vector is read at each of these positions.
SUFFIXES = [
    "", "I", "I think", "I think we", "Let me", "Sure", "Honestly", "Okay so",
    "The plan", "We should", "It is", "Yeah", "Right now", "Tomorrow", "That", "So",
]

def framed(persona: str, suffix: str) -> str:
    msgs = [
        {"role": "system", "content": persona},
        {"role": "user", "content": "Give me an update on the project."},
    ]
    s = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return s + suffix

dataset = [DatasetEntry(positive=framed(POS, s), negative=framed(NEG, s)) for s in SUFFIXES]
print(len(dataset), "contrastive pairs")
print("--- one positive example (tail) ---")
print(dataset[2].positive[-160:])

## 2. Extract the control vector

One pass over the pairs — no gradients, no training loop. This is why it's cheap.

In [ ]:
model.reset()  # make sure no previous control is active
hinglish_vector = ControlVector.train(model, tokenizer, dataset)

_layers = list(hinglish_vector.directions.keys())
print("control vector trained over", len(_layers), "layers")
print("per-layer direction shape:", hinglish_vector.directions[_layers[0]].shape)

## 3. Turn the dial — same prompt, three tones

`coeff > 0` → push toward Hinglish; `coeff < 0` → push toward formal English; `0` → the untouched model. If the text shifts with the number, the thesis holds.

In [ ]:
def generate(prompt: str, coeff: float) -> str:
    model.reset()
    if coeff != 0:
        model.set_control(hinglish_vector, coeff)
    msgs = [{"role": "user", "content": prompt}]
    ids = tokenizer.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to(model.device)
    out = model.generate(
        ids,
        max_new_tokens=80,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
    )
    model.reset()
    return tokenizer.decode(out[0, ids.shape[-1]:], skip_special_tokens=True).strip()

PROMPT = "Can you give me an update on the deployment?"
for c in [-2.0, 0.0, 2.0]:
    print(f"\n=========== coeff {c:+} ===========")
    print(generate(PROMPT, c))

## 4. Read the result + what's next

**Expected:** `+2` reads casual Hinglish ("haan bhai deployment ho gaya, testing chal rahi hai..."), `-2` reads formal English ("The deployment has been completed successfully..."), `0` is in between. That single vector *is* a reusable tone dial — no retraining.

**Tuning knobs if it's weak or breaks:**
- coeff too high (≥ ~3) → text degenerates; back off to 1.0–2.0.
- weak effect → widen the layer band (`range(-3, -22, -1)`) or add more contrastive pairs.

**Path into Vachan (later, not this notebook):**
1. Per-persona vectors: build the contrast from the persona's *own* anchors (Hinglish vs their English-translated anchors — we already store both) → a personalized tone dial.
2. Serve it: this needs *our* forward pass (Groq can't inject vectors). Run this 8B (or Sarvam-30B on 2×T4) behind vLLM/transformers on a serverless GPU, only for high-value personas that the PFS gate keeps failing — exactly the documented Path-B trigger.
3. The Fidelity Ring's neural cosine becomes the objective we tune the coeff against.